# Notebook 01 (Participant): Train + Generate with EngiOpt CGAN-2D

You will implement the full train-and-generate path and produce artifacts consumed by Notebook 02.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.

### Public exercise legend
- `PUBLIC FILL-IN CELL`: edit this cell directly.
- `CHECKPOINT`: run and verify before continuing.
- `IF YOU ARE STUCK`: use hint comments in the same cell.


## Standalone guide

This chapter is about **method integration under benchmark constraints**.
Success means reproducible artifacts and interpretable diagnostics, not only low training loss.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab


def pip_install(packages: list[str]):
    cmd = [sys.executable, '-m', 'pip', 'install', *packages]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)
BASE_PACKAGES = ['engibench[beams2d]', 'sqlitedict', 'matplotlib', 'tqdm', 'tyro', 'wandb']
ENGIOPT_GIT = 'git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt'

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    pip_install(BASE_PACKAGES)
    pip_install([ENGIOPT_GIT])

    try:
        import torch  # noqa: F401
    except Exception:
        pip_install(['torch', 'torchvision'])

    print('Dependency install complete.')
else:
    print('Skipping install (using current environment). Set FORCE_INSTALL=True to install here.')


## Part A: Setup

Lock down runtime, seeds, and artifact paths before writing model logic.


### EngiBench vs EngiOpt roles in this notebook

- EngiBench: defines data semantics, constraints, and simulator objective.
- EngiOpt: defines the generative model family and training dynamics.

Keep this separation explicit in your reasoning and reporting.


### Step 1 - Configure reproducible environment

Set all global controls once; downstream cells should rely on these values only.


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from engibench.problems.beams2d.v0 import Beams2D

try:
    from engiopt.cgan_2d.cgan_2d import Generator as EngiOptCGAN2DGenerator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Could not import engiopt model class. Run the bootstrap cell first; on Colab, restart runtime after install if needed.'
    ) from exc

USE_WANDB_ARTIFACTS = False
WANDB_PROJECT = 'dcc26-workshop'
WANDB_ENTITY = None
WANDB_ARTIFACT_NAME = 'dcc26_beams2d_generated_artifacts'
WANDB_ARTIFACT_ALIAS = 'latest'
WANDB_LOG_TRAINING = True


def resolve_artifact_dir(create: bool = False) -> Path:
    in_colab = 'google.colab' in sys.modules
    path = Path('/content/dcc26_artifacts') if in_colab else Path('workshops/dcc26/artifacts')
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


SEED = 7
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

DEVICE = th.device('cuda' if th.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ARTIFACT_DIR = resolve_artifact_dir(create=True)
print('artifact dir:', ARTIFACT_DIR)

CKPT_PATH = ARTIFACT_DIR / 'engiopt_cgan2d_generator_supervised.pt'
HISTORY_PATH = ARTIFACT_DIR / 'training_history.csv'
TRAIN_CURVE_PATH = ARTIFACT_DIR / 'training_curve.png'
LATENT_DIM = 32


### Step 2 - Build training slice from EngiBench dataset

Use a compact subset for workshop runtime; treat this as a pedagogical approximation, not final benchmark protocol.


In [ ]:
problem = Beams2D(seed=SEED)
train_ds = problem.dataset['train']
test_ds = problem.dataset['test']

condition_keys = problem.conditions_keys
print('condition keys:', condition_keys)

N_TRAIN = 512
subset_idx = np.random.default_rng(SEED).choice(len(train_ds), size=N_TRAIN, replace=False)

conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
designs_np = np.array(train_ds['optimal_design'])[subset_idx].astype(np.float32)
targets_np = (designs_np * 2.0) - 1.0

print('conditions shape:', conds_np.shape)
print('designs shape:', designs_np.shape)
print('target range:', float(targets_np.min()), 'to', float(targets_np.max()))


### Step 3 - Implement model setup (PUBLIC FILL-IN)

Instantiate generator, optimizer, and loss exactly once.

Success criteria:
- noise tensor shape is `(batch, LATENT_DIM)`,
- model output shape matches `problem.design_space.shape`.


In [ ]:
# PUBLIC FILL-IN CELL 01-A
# Goal: set up the EngiOpt generator + training primitives.

# START FILL ---------------------------------------------------------------
model = None
optimizer = None
criterion = None

def sample_noise(batch_size: int) -> th.Tensor:
    # Return standard normal latent vectors on DEVICE.
    raise NotImplementedError('Implement sample_noise')
# END FILL -----------------------------------------------------------------

if model is None or optimizer is None or criterion is None:
    raise RuntimeError('Define model, optimizer, and criterion in the START FILL block.')

# CHECKPOINT: validate latent shape and forward-pass shape
z_probe = sample_noise(4)
assert tuple(z_probe.shape) == (4, LATENT_DIM), f'Expected (4, {LATENT_DIM}), got {tuple(z_probe.shape)}'
cond_probe = th.tensor(conds_np[:4], dtype=th.float32, device=DEVICE)
with th.no_grad():
    pred_probe = model(z_probe, cond_probe)
assert tuple(pred_probe.shape[1:]) == tuple(problem.design_space.shape), (
    f'Output shape mismatch: expected tail {problem.design_space.shape}, got {tuple(pred_probe.shape[1:])}'
)
print('Checkpoint passed: model setup is consistent with problem representation.')


### Step 4 - Implement train/load logic (PUBLIC FILL-IN)

Track loss per epoch and persist checkpoint/history outputs.

Success criteria:
- training path writes checkpoint + history + curve,
- load path restores a checkpoint without retraining.


In [ ]:
TRAIN_FROM_SCRATCH = True
EPOCHS = 8
BATCH_SIZE = 64

# PUBLIC FILL-IN CELL 01-B
# Goal: train quickly for workshop runtime or load an existing checkpoint.

train_losses = []

if TRAIN_FROM_SCRATCH:
    # START FILL -----------------------------------------------------------
    # 1) Build DataLoader from conds_np + targets_np
    # 2) Run epoch loop with model.train()
    # 3) For each batch: predict, compute loss, backward, optimizer step
    # 4) Append epoch-average loss to train_losses
    # 5) Save checkpoint to CKPT_PATH
    # 6) Save history CSV to HISTORY_PATH
    # 7) Save training curve figure to TRAIN_CURVE_PATH
    raise NotImplementedError('Implement TRAIN_FROM_SCRATCH branch')
    # END FILL -------------------------------------------------------------
elif CKPT_PATH.exists():
    # START FILL -----------------------------------------------------------
    # Load checkpoint into model and, if available, load history CSV.
    raise NotImplementedError('Implement checkpoint load branch')
    # END FILL -------------------------------------------------------------
else:
    raise FileNotFoundError(f'Checkpoint not found at {CKPT_PATH}. Train first or provide checkpoint.')

# CHECKPOINT
assert CKPT_PATH.exists(), f'Missing checkpoint: {CKPT_PATH}'
assert HISTORY_PATH.exists(), f'Missing history CSV: {HISTORY_PATH}'
print('Checkpoint passed: train/load artifacts are ready for generation step.')


### Step 5 - Implement generation logic (PUBLIC FILL-IN)

Generate conditioned designs on held-out test conditions.

Success criteria:
- generated and baseline arrays are shape-compatible,
- condition records are JSON-serializable and aligned with samples.


In [ ]:
# PUBLIC FILL-IN CELL 01-C
# Goal: create generated designs + baseline designs + condition records.

N_SAMPLES = 24

# START FILL ---------------------------------------------------------------
# Suggested sequence:
# 1) sample indices from test_ds
# 2) build test_conds and baseline_designs
# 3) run model in eval/no_grad with sample_noise
# 4) map tanh output [-1,1] -> [0,1] and clip
# 5) build `conditions_records` as list[dict]
raise NotImplementedError('Implement generation block')
# END FILL -----------------------------------------------------------------

# CHECKPOINT
assert 'gen_designs' in locals(), 'Define gen_designs'
assert 'baseline_designs' in locals(), 'Define baseline_designs'
assert 'test_conds' in locals(), 'Define test_conds'
assert 'conditions_records' in locals(), 'Define conditions_records'
assert gen_designs.shape == baseline_designs.shape, 'Generated and baseline shapes must match'
assert len(conditions_records) == gen_designs.shape[0], 'conditions_records length mismatch'
print('Checkpoint passed: generation outputs are valid and aligned.')


### Step 6 - Implement artifact export (PUBLIC FILL-IN)

Notebook 02 expects these files as a strict handoff contract.
Treat artifact naming and file format as part of benchmark reproducibility.

Required files:
- `generated_designs.npy`
- `baseline_designs.npy`
- `conditions.json`


In [ ]:
# PUBLIC FILL-IN CELL 01-D
# Goal: export Notebook 02 handoff artifacts (plus optional extras).

# START FILL ---------------------------------------------------------------
# Required exports:
# - np.save(ARTIFACT_DIR / 'generated_designs.npy', gen_designs)
# - np.save(ARTIFACT_DIR / 'baseline_designs.npy', baseline_designs)
# - json dump of conditions_records -> conditions.json
# Recommended exports:
# - checkpoint (.pt), training_history.csv, training_curve.png
raise NotImplementedError('Implement artifact export block')
# END FILL -----------------------------------------------------------------

# CHECKPOINT
required_files = [
    ARTIFACT_DIR / 'generated_designs.npy',
    ARTIFACT_DIR / 'baseline_designs.npy',
    ARTIFACT_DIR / 'conditions.json',
]
missing = [str(f) for f in required_files if not f.exists()]
if missing:
    raise RuntimeError('Missing required artifacts:\\n' + '\\n'.join(missing))
print('Checkpoint passed: Notebook 02 handoff artifacts exist.')


### Step 7 - Quick visual QA

Use this for fast sanity checks only; final judgment comes from Notebook 02 simulator metrics.


In [ ]:
# Quick visual side-by-side snapshot
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(gen_designs[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'gen {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(baseline_designs[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'base {i}')
    axes[1, i].axis('off')

fig.tight_layout()
plt.show()


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Next

Proceed to Notebook 02 for physics-based evaluation and benchmark interpretation.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
